# Time series clustering
This notebook contains a comprehensive workflow for time series clustering analysis. It includes data preprocessing, feature selection, correlation analysis, and visualization. We apply both KMeans and hierarchical clustering (with various linkage methods), evaluates clustering quality using multiple metrics (such as silhouette and Calinski-Harabasz scores), and visualizes the results with heatmaps, scatter plots, radar charts, dendrograms, and cluster size distributions. The analysis is performed on extracted time series features from patient data.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt, find_peaks
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from timeit import default_timer as timer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV
from tslearn.clustering import TimeSeriesKMeans

ts_features_df = pd.read_pickle('../../data/processed/time_series_features.pkl')
ts_features_df

In [ ]:
# Convert subject_id and hadm_id to object type
ts_features_df['subject_id'] = ts_features_df['subject_id'].astype('object')
ts_features_df['hadm_id'] = ts_features_df['hadm_id'].astype('object')

# Take only numerical columns for correlation analysis
numerical_cols = ts_features_df.select_dtypes(include=[np.number])

print(f"Numerical columns: {numerical_cols.columns.tolist()}")

We begin by computing the correlation matrix of the DataFrame. This allows us to identify features that contain redundant information and could potentially mislead the clustering algorithms.

In [ ]:

# Plot the correlation matrix
fig, ax = plt.subplots(figsize=(20, 18), dpi=300)
correlation_matrix = numerical_cols.corr()

# Create a mask for the upper triangle
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))

sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 9}, ax=ax)
plt.title('Correlation Matrix of Time Series Features', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

Then we analyze the correlations and retain only one variable among those whose correlation exceeds the threshold. This reduces the number of features while preserving as much information as possible.

In [ ]:
# Keep only numerical columns
numerical_cols = ts_features_df.select_dtypes(include=[np.number])

threshold = 0.7

# Remove columns with correlation higher than threshold
corr_matrix = numerical_cols.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > threshold)]

# Print columns to be dropped
print(f"Columns to be dropped (correlation > {threshold}): {to_drop}")
print(f"Columns used for clustering: {[col for col in numerical_cols.columns if col not in to_drop]}")

reduced_df = numerical_cols.drop(columns=to_drop)

reduced_df

In [ ]:
# Re plot the correlation matrix after removing correlated features
fig, ax = plt.subplots(figsize=(20, 18), dpi=300)
correlation_matrix = reduced_df.corr()
# Create a mask for the upper triangle
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))   
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt=".2f", cmap='coolwarm',
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            annot_kws={"size": 9}, ax=ax)
plt.title('Correlation Matrix of Reduced Time Series Features', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()

## Feature-based clustering
We clustered the data using two different approaches:
1. K-means algorithm
2. Hierarchical clustering

For each method, we first tuned the algorithm’s parameters to find the best configuration, and then ran the clustering with these optimal values.

### K-Means

In [ ]:
# Standardize the data
scaler = StandardScaler()
scaled_data = scaler.fit_transform(reduced_df)

The following code applies the elbow method to identify the optimal number of clusters for K‑Means clustering.
It fits K‑Means models for a range of cluster counts (k = 2 to 49) and records the inertia (sum of squared errors, SSE) for each k. The relative drop in inertia between successive values of k is then computed to locate the “elbow” point—where the decrease in SSE is largest and further increases in k yield diminishing returns.
This elbow point is taken as the most suitable number of clusters. The code visualizes the SSE and highlights the selected k in a plot, guiding the choice of cluster count for subsequent analysis.

In [ ]:
# Apply the elbow method to find the optimal number of clusters and plot the best k
inertia = []
K = range(2, 50)
for k in K:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(scaled_data)
    inertia.append(kmeans.inertia_)

# Find the elbow: largest relative drop in inertia (SSE)
rel_drop = np.diff(inertia) / inertia[:-1]
if len(rel_drop) > 0:
    best_k = K[np.argmin(rel_drop) + 1]
else:
    best_k = K[0]

plt.figure(figsize=(20, 16), dpi=200)
plt.plot(K, inertia, 'bo-', label='SSE')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.xticks(K)
plt.grid(True)
plt.axvline(best_k, color='red', linestyle='--', label=f'Best k (elbow): {best_k}')
plt.legend(fontsize=14)
plt.show()

The elbow method shows that the best value for K should be $K=4$, but, looking better at the plots it is possible to notice that the elbow is in the range $[9,21]$, so we refined the search inside that area. To do so we define three kind of scorer:
1) Shilouette scorer: Scorer that uses the Shilouette score
2) MSS scorer: Scorer that uses the Calinski-Harabasz score
3) Mixed scorer: Scorer that combine both (1) and (2)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.model_selection import GridSearchCV
from sklearn import metrics
from sklearn.metrics import silhouette_score, make_scorer

def silhouette_scorer(estimator, X):
    """
    Computes the Silhouette Score for clustering results.

    The Silhouette Score measures how similar each sample is to its own cluster compared to other clusters.
    It ranges from -1 (incorrect clustering) to +1 (highly dense clustering), with values near 0 indicating overlapping clusters.

    Args:
        estimator: Fitted clustering estimator with a predict method (e.g., KMeans).
        X (array-like): Feature data to cluster.

    Returns:
        float: Silhouette Score if at least 2 clusters are present, otherwise -1.0.
    """
    clusters = estimator.predict(X)
    if len(set(clusters)) < 2:
        return -1.0
    return silhouette_score(X, clusters)

def mss_scorer(estimator, X):
    """
    Computes the Calinski-Harabasz (MSS) Score for clustering results.

    The Calinski-Harabasz Score evaluates cluster validity based on the ratio of between-cluster dispersion to within-cluster dispersion.
    Higher values indicate better-defined clusters.

    Args:
        estimator: Fitted clustering estimator with a predict method (e.g., KMeans).
        X (array-like): Feature data to cluster.

    Returns:
        float: Calinski-Harabasz Score if at least 2 clusters are present, otherwise -1.0.
    """
    clusters = estimator.predict(X)
    if len(set(clusters)) < 2:
        return -1.0
    return metrics.calinski_harabasz_score(X, clusters)

def combined_scorer(estimator, X):
    """
    Computes a composite clustering quality score by combining Silhouette Score, Calinski-Harabasz Score, and Davies-Bouldin Score.

    This metric averages the Silhouette Score and Calinski-Harabasz Score (where higher is better)
    and subtracts the Davies-Bouldin Score (where lower is better), providing a balanced assessment of clustering quality.

    Args:
        estimator: Fitted clustering estimator with a predict method (e.g., KMeans).
        X (array-like): Feature data to cluster.

    Returns:
        float: Combined score if at least 2 clusters are present, otherwise -1.0.
    """
    clusters = estimator.predict(X)
    if len(set(clusters)) < 2:
        return -1.0
    sil_score = silhouette_score(X, clusters)
    mss_score = metrics.calinski_harabasz_score(X, clusters)
    db_score = metrics.davies_bouldin_score(X, clusters)
    combined_score = (sil_score + mss_score - db_score) / 3
    return combined_score

We evaluate K-Means clustering performance across a range of cluster counts (k = 2 to 21). For each value of k, we fit a K-Means model and compute four key clustering metrics:
- SSE (inertia): Within-cluster sum of squared distances (lower is better).
- MSS (Calinski–Harabasz Score): Degree of cluster separation (higher is better).
- Silhouette Score: How well samples fit within their assigned clusters (higher is better).
- Davies–Bouldin Score: Average similarity between clusters (lower is better).
All results are stored in a DataFrame for subsequent analysis and comparison, supporting the choice of the most appropriate number of clusters.

In [ ]:
# Evaluate KMeans clustering for a range of cluster numbers (k = 2 to 21)
results = []

for k in range(2, 22):
    # Initialize and fit KMeans with specified k
    kmeans = KMeans(n_clusters=k, n_init=10, max_iter=500, random_state=42)
    kmeans.fit(scaled_data)
    labels = kmeans.labels_

    # Compute clustering quality metrics:
    sse = kmeans.inertia_  # Sum of squared errors (inertia)
    silhouette = silhouette_score(scaled_data, labels)  # Silhouette Score
    db = metrics.davies_bouldin_score(scaled_data, labels)  # Davies-Bouldin Score
    mss = metrics.calinski_harabasz_score(scaled_data, labels)  # Calinski-Harabasz (MSS) Score

    # Store results for each k
    results.append([k, sse, mss, silhouette, db])

# Create a DataFrame to summarize the results for all k
results_df = pd.DataFrame(results, columns=["K", "SSE", "MSS", "Silhouette", "DB_score"])
print(results_df)

The main idea was to identify and use the value of $K$ that stabilizes the majority of the clustering scores, thereby providing a robust and consistent choice for the number of clusters. However, since such a stabilizing value of $K$ does not emerge clearly from our results, we instead proceed by relying on the silhouette index as the primary scoring metric to guide the remainder of our analysis and to select the most appropriate clustering configuration. To find the best parameters, se execute a grid search using a 5-fold cross validation and then taking the best KMeans model.

In [ ]:
# Define the parameter grid for KMeans hyperparameter tuning
param_grid = {
    'n_clusters': range(4, 22),           # Try cluster counts from 4 to 21
    'init': ['k-means++', 'random'],      # Test both initialization methods
    'n_init': [10]                        # Fix number of initializations for stability
}

# Initialize the KMeans estimator (random_state ensures reproducibility)
kmeans = KMeans(random_state=42)

# Set up the grid search with cross-validation
# - n_jobs=-1: use all CPU cores for speed
# - verbose=1: print progress messages
# - cv=5: 5-fold cross-validation
# - scoring=silhouette_scorer: use custom silhouette score function
grid_search = GridSearchCV(
    kmeans, 
    param_grid, 
    cv=5, 
    scoring=silhouette_scorer, 
    n_jobs=-1,
    verbose=1
)

print("Starting Grid Search...")

# Fit the grid search to the standardized data
grid_search.fit(scaled_data)

# Print the best silhouette score and corresponding parameters
print(f"\nBest Silhouette Score: {grid_search.best_score_:.3f}")
print(f"Best Parameters: {grid_search.best_params_}")

# Retrieve the best KMeans model found by grid search
best_kmeans = grid_search.best_estimator_

Now that we have found the best value for $K$, we can run the KMeans algorithm using the best parameters found with the grid search using a 5-fold cross validation.

In [ ]:
# Run Kmeans with the best parameters
final_kmeans = KMeans(
    n_clusters=best_kmeans.n_clusters,
    init=best_kmeans.init,
    n_init=best_kmeans.n_init,
    random_state=42
)
final_kmeans.fit(scaled_data)

In [ ]:
# Add the cluster labels to the original dataframe
ts_features_df['cluster'] = final_kmeans.labels_

In [ ]:
# Plot KMeans cluster centers (profile plot)

# Use KMeans cluster labels and centers
cluster_labels = final_kmeans.labels_
ts_features_df['cluster_kmeans'] = cluster_labels

# Use reduced_df columns and scaled_data for features and centers
feature_names = reduced_df.columns.tolist()
X = scaled_data
n_clusters = final_kmeans.n_clusters

# Calculate centers for each cluster (should match final_kmeans.cluster_centers_)
centers = []
for cluster_id in range(n_clusters):
    cluster_mask = cluster_labels == cluster_id
    cluster_center = X[cluster_mask].mean(axis=0)
    centers.append(cluster_center)
centers = np.array(centers)

# Plot cluster centers
plt.figure(figsize=(25, 10), dpi=300)
for i in range(len(centers)):
    plt.plot(centers[i], marker='o', label=f'Cluster {i + 1}', linewidth=2, markersize=8)

plt.tick_params(axis='both', which='major', labelsize=10)
plt.xticks(range(len(feature_names)), feature_names, fontsize=12, rotation=45, ha='right')
plt.xlabel('Features', fontsize=14, fontweight='bold')
plt.ylabel('Scaled Feature Values', fontsize=14, fontweight='bold')
plt.title('KMeans Cluster Centers Profile', fontsize=16, fontweight='bold')
plt.legend(fontsize=12, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from math import pi
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import matplotlib.pyplot as plt

# Use the correct feature names and cluster centers
feature_names = reduced_df.columns.tolist()
centers = final_kmeans.cluster_centers_
N = len(feature_names)
n_clusters = centers.shape[0]

# Normalize the centers for radar chart
scaler_radar = MinMaxScaler()
centers_normalized = scaler_radar.fit_transform(centers.T).T

# Create a color palette with exactly n_clusters colors
if n_clusters <= 10:
    colors_list = plt.cm.tab10(np.linspace(0, 1, 10))[:n_clusters]
elif n_clusters <= 20:
    colors_list = plt.cm.tab20(np.linspace(0, 1, 20))[:n_clusters]
else:
    # If more than 20 clusters, use a continuous colormap
    colors_list = plt.cm.nipy_spectral(np.linspace(0, 1, n_clusters))

# Create ONE radar plot with all clusters overlapped
fig = plt.figure(figsize=(20, 20), dpi=300)
ax = plt.subplot(111, polar=True)

# Calculate angles for each feature
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]  # Complete the circle

# Draw one axis per variable and add labels with better visibility
plt.xticks(angles[:-1], feature_names, color='black', size=12, fontweight='bold')

# Plot all clusters on the same radar chart
for i in range(len(centers_normalized)):
    values = centers_normalized[i].tolist()
    values += values[:1]  # Complete the circle

    # Plot data with unique color for each cluster
    ax.plot(angles, values, linewidth=3, linestyle='solid',
            label=f'Cluster {i+1}', color=colors_list[i])

    # Fill area with the same color
    ax.fill(angles, values, alpha=0.15, color=colors_list[i])

# Set y-axis limits
ax.set_ylim(0, 1)

# Increase y-axis tick label size
ax.tick_params(axis='y', labelsize=8)

# Add title
ax.set_title('All Clusters Profile Comparison', size=18, fontweight='bold', pad=40)

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=13,
           title='Clusters', title_fontsize=14, frameon=True, shadow=True)

# Add grid
ax.grid(True, linestyle='--', linewidth=0.7, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Plot all the combinations of the original features colored by cluster (ward) in a single plot
import itertools
# Include 'age' only if not already present
features_to_plot = numerical_cols.columns.tolist()
if 'age' not in features_to_plot:
    features_to_plot.append('age')
feature_combinations = list(itertools.combinations(features_to_plot, 2))
num_combinations = len(feature_combinations)
if num_combinations == 0:
    print("Not enough features to create scatter plot combinations (need at least 2 features).")
else:
    cols = 3
    rows = max(1, (num_combinations + cols - 1) // cols)  # Ensure at least 1 row
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
    axes = np.atleast_2d(axes)  # Ensure axes is always 2D for consistent flattening
    
    # Create a consistent color palette mapping for all clusters
    unique_clusters = sorted(ts_features_df['cluster'].unique())
    colors = sns.color_palette('tab20', n_colors=len(unique_clusters))  # Supports up to 20 colors
    palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}
    
    for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_combinations):
        sns.scatterplot(
            data=ts_features_df, 
            x=feat_x, 
            y=feat_y, 
            hue='cluster', 
            palette=palette_dict,  # Use the dictionary for consistent colors
            ax=ax,
            alpha=0.7,
            legend=False  # Disable individual legends
        )
        ax.set_title(f'Scatter Plot of {feat_x} vs {feat_y}', fontsize=12)
    
    # Remove any unused subplots
    for i in range(len(feature_combinations), len(axes.flatten())):
        fig.delaxes(axes.flatten()[i])
    
    # Create a single shared legend using the same palette
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                          markersize=10, alpha=0.7, label=f'Cluster {c}') 
               for c in unique_clusters]
    
    fig.legend(handles=handles, title='Cluster', loc='upper right', 
               bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space for the legend
    plt.show()

In [ ]:
# Select all numerical columns for the scatter matrix (not just those used for clustering)
vars_scatter = numerical_cols.columns.tolist()

# Prepare the DataFrame for plotting
df_scatter = ts_features_df.copy()
df_scatter["Cluster"] = df_scatter["cluster"]  # Ensure the cluster column is named 'Cluster' for hue

# Apply log10(x + 1) transformation to all numeric columns to avoid log(0)
for col in vars_scatter:
    df_scatter[col] = np.log10(df_scatter[col] + 1)

sns.pairplot(
    df_scatter[vars_scatter + ["Cluster"]],
    hue="Cluster",
    palette="tab10",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4, "s": 25}
)

plt.suptitle("Scatter Matrix (all numeric features, log10(x+1) scale)", y=1.02)
plt.show()

In [ ]:
# Use only the variables used for clustering (e.g., reduced_df.columns)
vars_scatter = reduced_df.columns.tolist()

# Prepare the DataFrame for plotting
df_scatter = ts_features_df.copy()
df_scatter["Cluster"] = ts_features_df["cluster"]

# Apply log10(x + 1) transformation to the selected variables
for col in vars_scatter:
    df_scatter[col] = np.log10(df_scatter[col] + 1)

sns.pairplot(
    df_scatter[vars_scatter + ["Cluster"]],
    hue="Cluster",
    palette="tab10",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4, "s": 25}
)

plt.suptitle("Scatter Matrix (clustering features, log10(x+1) scale)", y=1.02)
plt.show()

In [ ]:
# Plot the cluster size distribution
cluster_counts = ts_features_df['cluster'].value_counts().sort_index()
plt.figure(figsize=(15, 10), dpi=300)
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette='tab10')
plt.xlabel('Cluster', fontsize=14, fontweight='bold')
plt.ylabel('Number of Time Series', fontsize=14, fontweight='bold')
plt.title('KMeans Cluster Size Distribution', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
for i, count in enumerate(cluster_counts.values):
    plt.text(i, count - count * 0.1, str(count), ha='center', fontsize=10, color='black')

From the plot above, Cluster 3 ($n = 128$) and Cluster 15 ($n = 103$) emerge as the most populous groups. 

In ECG analysis, such large clusters typically correspond to the most standard or “normal” heartbeat morphologies (e.g., normal sinus rhythm) that dominate the patient population. 
By contrast, Cluster 19 contains a single time series ($n = 1$). This “singleton” cluster is of particular diagnostic interest because it likely reflects a unique anomaly, a severe artifact, or a rare pathological event that is morphologically distinct from all other signals in the dataset.
The majority of the remaining clusters (e.g., Clusters 1, 2, 7, 9, 11, 12, and 18) each contain between 50 and 90 samples, and can be interpreted as common variations or specific subtypes of arrhythmias that are distinct yet well represented in the data.

### Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import linkage, cophenet
from scipy.spatial.distance import pdist
def get_cophenetic_coefficients(X, methods=['single', 'complete', 'average', 'ward']) -> pd.DataFrame:
    """ 
    Calculate cophenetic correlation coefficients for various hierarchical clustering methods.

    This function iterates through specified linkage methods, computes the linkage matrix, 
    and determines how faithfully the dendrogram preserves the pairwise distances 
    between the original data points.

    Parameters
    ----------
    X : array-like
        The input data to cluster. It is recommended to scale this data 
        before passing it to the function.
    methods : list of str, optional
        A list of linkage methods to evaluate. 
        Defaults to ['single', 'complete', 'average', 'ward'].

    Returns
    -------
    pd.DataFrame
        A DataFrame where columns correspond to the clustering methods and 
        values represent the cophenetic correlation coefficients.
    """
    cophenetic_coefficients_df = pd.DataFrame()

    for method in methods:
        # Compute the linkage matrix
        Z = linkage(X, method)
        
        # Calculate the cophenetic correlation coefficient
        # pdist calculates the pairwise Euclidean distances of the input data
        c, _ = cophenet(Z, pdist(X, metric='euclidean'))
        
        cophenetic_coefficients_df[method] = [c]
    
    return cophenetic_coefficients_df

# Initialize the scaler
hc_scaler = MinMaxScaler()
scaled_data = hc_scaler.fit_transform(reduced_df)

# Use the data that has already been scaled using MinMaxScaler
get_cophenetic_coefficients(scaled_data)

To choose the best linkage method for the Hierarchical Clustering, we first analyse the dendrograms obtained by the algorithm

In [ ]:
from scipy.cluster.hierarchy import dendrogram

methods = ['single', 'complete', 'average', 'ward']

fig, axs = plt.subplots(2, 2, figsize=(14, 14), dpi=150)

for ax, method in zip(axs.flatten(), methods):
    data_dist = pdist(scaled_data, metric='euclidean')
    data_link = linkage(data_dist, method=method, metric='euclidean')
    res = dendrogram(data_link, truncate_mode='lastp', ax=ax, p=20)
    ax.set_title(f'{method} linkage', fontsize=14, fontweight='bold')
    ax.tick_params(labelrotation=90, labelsize=10)
    ax.set_xlabel('Sample index', fontsize=11, fontweight='bold')
    ax.set_ylabel('Distance', fontsize=11, fontweight='bold')

fig.suptitle("Dendrogram for different hierarchical clustering methods", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

From the figure, we can see that the single and average methods produce rather unusual dendrograms, whereas the complete and Ward linkage methods yield more interpretable dendrograms. Therefore, we focus our analysis on these two methods calculating the scores to make a more precise decision.

In [ ]:
from sklearn.metrics import silhouette_score, calinski_harabasz_score
from scipy.cluster.hierarchy import cophenet, linkage
from scipy.spatial.distance import pdist
import numpy as np
import matplotlib.pyplot as plt

# Linkage methods to compare
methods_to_compare = ['ward', 'complete']
k_range = range(2, 22)

results = {}

from sklearn.metrics import pairwise_distances

def intra_inter_cluster_distances(X, labels):
    """Compute mean intra-cluster and inter-cluster distances."""
    unique_labels = np.unique(labels)
    n_clusters = len(unique_labels)
    intra_dists = []
    inter_dists = []
    for label in unique_labels:
        cluster_points = X[labels == label]
        if len(cluster_points) > 1:
            dists = pairwise_distances(cluster_points)
            # Exclude diagonal (self-distances)
            intra = dists[np.triu_indices_from(dists, k=1)].mean()
            intra_dists.append(intra)
        else:
            intra_dists.append(0.0)
    # Inter-cluster: mean distance between cluster centers
    centers = np.array([X[labels == label].mean(axis=0) for label in unique_labels])
    if n_clusters > 1:
        inter = pairwise_distances(centers)
        inter_mean = inter[np.triu_indices_from(inter, k=1)].mean()
    else:
        inter_mean = 0.0
    return np.mean(intra_dists), inter_mean

# Add to results collection
for method in methods_to_compare:
    print(f"\n{'='*60}")
    print(f"Metodo: {method.upper()}")
    print(f"{'='*60}")
    
    data_dist = pdist(scaled_data, metric='euclidean')
    data_link = linkage(data_dist, method=method, metric='euclidean')
    
    silhouette_scores = []
    calinski_harabasz_scores = []
    intra_dists = []
    inter_dists = []
    
    for k in k_range:
        from scipy.cluster.hierarchy import cut_tree
        cluster_labels = cut_tree(data_link, n_clusters=k).reshape(-1)
        
        sil_score = silhouette_score(scaled_data, cluster_labels, metric='euclidean')
        ch_score = calinski_harabasz_score(scaled_data, cluster_labels)
        intra, inter = intra_inter_cluster_distances(scaled_data, cluster_labels)
        
        silhouette_scores.append(sil_score)
        calinski_harabasz_scores.append(ch_score)
        intra_dists.append(intra)
        inter_dists.append(inter)
        
        print(f"k={k}: Silhouette={sil_score:.4f}, CH={ch_score:.2f}, Intra={intra:.4f}, Inter={inter:.4f}")
    
    coph_corr, coph_dists = cophenet(data_link, data_dist)
    print(f"\nCophenetic Correlation Coefficient: {coph_corr:.4f}")
    
    results[method] = {
        'silhouette': silhouette_scores,
        'calinski_harabasz': calinski_harabasz_scores,
        'intra': intra_dists,
        'inter': inter_dists,
        'cophenetic': coph_corr,
        'linkage': data_link
    }

# Update plots to include intra/inter cluster distances
fig, axes = plt.subplots(3, 2, figsize=(18, 16), dpi=300)

colors = {'ward': 'blue', 'average': 'green', 'complete': 'red'}

# Plot 1: Silhouette Score
for method in methods_to_compare:
    axes[0, 0].plot(k_range, results[method]['silhouette'], 
                    marker='o', linewidth=2, markersize=6, 
                    label=f'{method} (max={max(results[method]["silhouette"]):.4f})',
                    color=colors[method])
axes[0, 0].set_xlabel('Number of Clusters', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Silhouette Score', fontsize=12, fontweight='bold')
axes[0, 0].set_title('Silhouette Score Comparison', fontsize=14, fontweight='bold')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# Plot 2: Calinski-Harabasz Score
for method in methods_to_compare:
    axes[0, 1].plot(k_range, results[method]['calinski_harabasz'], 
                    marker='s', linewidth=2, markersize=6, 
                    label=f'{method} (max={max(results[method]["calinski_harabasz"]):.2f})',
                    color=colors[method])
axes[0, 1].set_xlabel('Number of Clusters', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Calinski-Harabasz Score', fontsize=12, fontweight='bold')
axes[0, 1].set_title('Calinski-Harabasz Score Comparison', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# Plot 3: Intra-cluster distance
for method in methods_to_compare:
    axes[1, 0].plot(k_range, results[method]['intra'], 
                    marker='^', linewidth=2, markersize=6, 
                    label=f'{method}',
                    color=colors[method])
axes[1, 0].set_xlabel('Number of Clusters', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Mean Intra-cluster Distance', fontsize=12, fontweight='bold')
axes[1, 0].set_title('Intra-cluster Distance', fontsize=14, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# Plot 4: Inter-cluster distance
for method in methods_to_compare:
    axes[1, 1].plot(k_range, results[method]['inter'], 
                    marker='v', linewidth=2, markersize=6, 
                    label=f'{method}',
                    color=colors[method])
axes[1, 1].set_xlabel('Number of Clusters', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Mean Inter-cluster Distance', fontsize=12, fontweight='bold')
axes[1, 1].set_title('Inter-cluster Distance', fontsize=14, fontweight='bold')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].legend()

# Plot 5: Normalized Comparison
for method in methods_to_compare:
    sil_norm = np.array(results[method]['silhouette'])
    axes[2, 0].plot(k_range, sil_norm, 
                    marker='o', linewidth=2, markersize=6, 
                    label=f'{method} - Silhouette',
                    color=colors[method], linestyle='-')
    ch_norm = np.array(results[method]['calinski_harabasz']) / np.max(results[method]['calinski_harabasz'])
    axes[2, 0].plot(k_range, ch_norm, 
                    marker='s', linewidth=2, markersize=6, 
                    label=f'{method} - CH (normalized)',
                    color=colors[method], linestyle='--')
axes[2, 0].set_xlabel('Number of Clusters', fontsize=12, fontweight='bold')
axes[2, 0].set_ylabel('Normalized Score', fontsize=12, fontweight='bold')
axes[2, 0].set_title('Metrics Comparison (Normalized)', fontsize=14, fontweight='bold')
axes[2, 0].grid(True, alpha=0.3)
axes[2, 0].legend(fontsize=9)

# Plot 6: Summary with optimal k
optimal_ks = {}
for method in methods_to_compare:
    optimal_k_sil = k_range[np.argmax(results[method]['silhouette'])]
    optimal_k_ch = k_range[np.argmax(results[method]['calinski_harabasz'])]
    optimal_ks[method] = {'silhouette': optimal_k_sil, 'calinski_harabasz': optimal_k_ch}

summary_text = "OPTIMAL K SUMMARY\n" + "="*40 + "\n\n"
for method in methods_to_compare:
    summary_text += f"{method.upper()}:\n"
    summary_text += f"  Cophenetic Coeff: {results[method]['cophenetic']:.4f}\n"
    summary_text += f"  Optimal k (Silhouette): {optimal_ks[method]['silhouette']}\n"
    summary_text += f"  Optimal k (CH Index): {optimal_ks[method]['calinski_harabasz']}\n"
    summary_text += f"  Max Silhouette: {max(results[method]['silhouette']):.4f}\n"
    summary_text += f"  Max CH Score: {max(results[method]['calinski_harabasz']):.2f}\n"
    summary_text += f"  Min Intra-cluster: {min(results[method]['intra']):.4f}\n"
    summary_text += f"  Max Inter-cluster: {max(results[method]['inter']):.4f}\n\n"

axes[2, 1].text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
                verticalalignment='center', transform=axes[2, 1].transAxes)
axes[2, 1].axis('off')

plt.tight_layout()
plt.show()

print(f"\n{'='*60}")
print("FINAL SUMMARY")
print(f"{'='*60}")
print(summary_text)

### Plot with complete linkage

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Perform hierarchical clustering with complete linkage
data_dist = pdist(scaled_data, metric='euclidean')
data_link = linkage(data_dist, method='complete')

# Get the best k for the average method from previous analysis
avg_best_k = results['complete']['silhouette'].index(max(results['complete']['silhouette'])) + 2  # +2 because k starts from 2

print(f"Best k for average linkage based on silhouette score: {avg_best_k}")

n_clusters = avg_best_k
ts_features_df['cluster'] = fcluster(data_link, t=n_clusters, criterion='maxclust')

In [ ]:
# Plot the dendogram with the cut to the number of clusters
plt.figure(figsize=(12, 8), dpi=150)
dendrogram(data_link, truncate_mode='lastp', p=n_clusters, color_threshold=None)
plt.title('Dendrogram with Complete Linkage', fontsize=14, fontweight='bold')
plt.ylabel('Distance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot cluster size distribution
cluster_counts = ts_features_df['cluster'].value_counts().sort_index()
plt.figure(figsize=(15, 10), dpi=300)
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette='tab10')
plt.xlabel('Cluster', fontsize=14, fontweight='bold')
plt.ylabel('Number of Time Series', fontsize=14, fontweight='bold')
plt.title('Hierarchical Clustering (Complete Linkage) Cluster Size Distribution', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
for i, count in enumerate(cluster_counts.values):
    plt.text(i, count - count * 0.1, str(count), ha='center', fontsize=10, color='black')   
plt.show()

The distribution from the complete linkage method exhibited a high degree of heterogeneity and imbalance. This method produced several extremely small clusters, indicating that it isolated outliers instead of grouping them into broader categories. Specifically, Cluster 1 (n=4), Cluster 17 (n=8), and Cluster 20 (n=1) contained minimal data. Most of the data was heavily concentrated in just a few large groups, with Cluster 6 (n=134) and Cluster 5 (n=116) dominating the distribution. This fragmentation implies that complete linkage may be sensitive to noise in the dataset, resulting in ”singleton” or near-singleton clusters that are not statistically useful for further analysis. In contrast, the Ward linkage method yielded a more balanced and structurally consistent partition of the data.

In [ ]:

# Include 'age' only if not already present
features_to_plot = numerical_cols.columns.tolist()
if 'age' not in features_to_plot:
    features_to_plot.append('age')
feature_combinations = list(itertools.combinations(features_to_plot, 2))
num_combinations = len(feature_combinations)
if num_combinations == 0:
    print("Not enough features to create scatter plot combinations (need at least 2 features).")
else:
    cols = 3
    rows = max(1, (num_combinations + cols - 1) // cols)  # Ensure at least 1 row
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
    axes = np.atleast_2d(axes)  # Ensure axes is always 2D for consistent flattening
    
    # Create a consistent color palette mapping for all clusters
    unique_clusters = sorted(ts_features_df['cluster'].unique())
    colors = sns.color_palette('tab20', n_colors=len(unique_clusters))  # Supports up to 20 colors
    palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}
    
    for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_combinations):
        sns.scatterplot(
            data=ts_features_df, 
            x=feat_x, 
            y=feat_y, 
            hue='cluster', 
            palette=palette_dict,  # Use the dictionary for consistent colors
            ax=ax,
            alpha=0.7,
            legend=False  # Disable individual legends
        )
        ax.set_title(f'Scatter Plot of {feat_x} vs {feat_y}', fontsize=12)
    
    # Remove any unused subplots
    for i in range(len(feature_combinations), len(axes.flatten())):
        fig.delaxes(axes.flatten()[i])
    
    # Create a single shared legend using the same palette
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                          markersize=10, alpha=0.7, label=f'Cluster {c}') 
               for c in unique_clusters]
    
    fig.legend(handles=handles, title='Cluster', loc='upper right', 
               bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)
    
    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space for the legend
    plt.show()

In [ ]:
# Select all numerical columns for the scatter matrix (not just those used for clustering)
vars_scatter = numerical_cols.columns.tolist()

# Prepare the DataFrame for plotting
df_scatter = ts_features_df.copy()
df_scatter["Cluster"] = df_scatter["cluster"]  # Ensure the cluster column is named 'Cluster' for hue

# Apply log10(x + 1) transformation to all numeric columns to avoid log(0)
for col in vars_scatter:
    df_scatter[col] = np.log10(df_scatter[col] + 1)

sns.pairplot(
    df_scatter[vars_scatter + ["Cluster"]],
    hue="Cluster",
    palette="tab10",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4, "s": 25}
)

plt.suptitle("Scatter Matrix (all numeric features, log10(x+1) scale)", y=1.02)
plt.show()

In [ ]:
# Use only the variables used for clustering (e.g., reduced_df.columns)
vars_scatter = reduced_df.columns.tolist()

# Prepare the DataFrame for plotting
df_scatter = ts_features_df.copy()
df_scatter["Cluster"] = ts_features_df["cluster"]

# Apply log10(x + 1) transformation to the selected variables
for col in vars_scatter:
    df_scatter[col] = np.log10(df_scatter[col] + 1)

sns.pairplot(
    df_scatter[vars_scatter + ["Cluster"]],
    hue="Cluster",
    palette="tab10",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4, "s": 25}
)

plt.suptitle("Scatter Matrix (clustering features, log10(x+1) scale)", y=1.02)
plt.show()

By analyzing the clusters obtained with the complete linkage method, we observe that it is difficult to identify clear, well-separated groups. Several points are mixed with points from other clusters, and in some cases, the high density of points likely contributes to suboptimal clustering. To mitigate this issue, we could either modify the feature set or reconsider the methods applied during the data understanding phase.

## Plot with ward

In [ ]:
from scipy.cluster.hierarchy import fcluster

# Perform hierarchical clustering with ward
data_dist = pdist(scaled_data, metric='euclidean')
data_link = linkage(data_dist, method='ward')
# Get the best k for the average method from previous analysis
ward_best_k = results['ward']['silhouette'].index(max(results['ward']['silhouette'])) + 2  

print(f"Best k for Ward linkage based on silhouette score: {ward_best_k}")
n_clusters = ward_best_k
ts_features_df['cluster'] = fcluster(data_link, t=n_clusters, criterion='maxclust')

In [ ]:
# Plot the dendogram with the cut to the number of clusters
plt.figure(figsize=(12, 8), dpi=150)
dendrogram(data_link, truncate_mode='lastp', p=n_clusters, color_threshold=None)
plt.title('Dendrogram with Ward Linkage', fontsize=14, fontweight='bold')
plt.ylabel('Distance', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Plot all the combinations of the original features colored by cluster (complete linkage) in a single plot

import itertools

# Ensure 'cluster_complete' column exists (assigned after complete linkage clustering)
if 'cluster_complete' not in ts_features_df.columns:
    ts_features_df['cluster_complete'] = ts_features_df['cluster']

# Include 'age' only if not already present
features_to_plot = numerical_cols.columns.tolist()
if 'age' not in features_to_plot:
    features_to_plot.append('age')
feature_combinations = list(itertools.combinations(features_to_plot, 2))
num_combinations = len(feature_combinations)
if num_combinations == 0:
    print("Not enough features to create scatter plot combinations (need at least 2 features).")
else:
    cols = 3
    rows = max(1, (num_combinations + cols - 1) // cols)  # Ensure at least 1 row
    fig, axes = plt.subplots(rows, cols, figsize=(20, rows * 5), dpi=300)
    axes = np.atleast_2d(axes)  # Ensure axes is always 2D for consistent flattening

    # Create a consistent color palette mapping for all clusters
    unique_clusters = sorted(ts_features_df['cluster_complete'].unique())
    colors = sns.color_palette('tab20', n_colors=len(unique_clusters))  # Supports up to 20 colors
    palette_dict = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

    for ax, (feat_x, feat_y) in zip(axes.flatten(), feature_combinations):
        sns.scatterplot(
            data=ts_features_df, 
            x=feat_x, 
            y=feat_y, 
            hue='cluster_complete', 
            palette=palette_dict,  # Use the dictionary for consistent colors
            ax=ax,
            alpha=0.7,
            legend=False  # Disable individual legends
        )
        ax.set_title(f'Scatter Plot of {feat_x} vs {feat_y} (Complete Linkage)', fontsize=12)

    # Remove any unused subplots
    for i in range(len(feature_combinations), len(axes.flatten())):
        fig.delaxes(axes.flatten()[i])

    # Create a single shared legend using the same palette
    handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=palette_dict[c], 
                          markersize=10, alpha=0.7, label=f'Cluster {c}') 
               for c in unique_clusters]

    fig.legend(handles=handles, title='Cluster (Complete Linkage)', loc='upper right', 
               bbox_to_anchor=(0.99, 0.99), fontsize=10, title_fontsize=12)

    plt.tight_layout(rect=[0, 0, 0.95, 1])  # Leave space for the legend
    plt.show()

In [ ]:
# Select all numerical columns for the scatter matrix (not just those used for clustering)
vars_scatter = numerical_cols.columns.tolist()

# Prepare the DataFrame for plotting
df_scatter = ts_features_df.copy()
df_scatter["Cluster"] = df_scatter["cluster"]  # Ensure the cluster column is named 'Cluster' for hue

# Apply log10(x + 1) transformation to all numeric columns to avoid log(0)
for col in vars_scatter:
    df_scatter[col] = np.log10(df_scatter[col] + 1)

sns.pairplot(
    df_scatter[vars_scatter + ["Cluster"]],
    hue="Cluster",
    palette="tab10",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4, "s": 25}
)

plt.suptitle("Scatter Matrix (all numeric features, log10(x+1) scale)", y=1.02)
plt.show()

In [ ]:
# Use only the variables used for clustering (e.g., reduced_df.columns)
vars_scatter = reduced_df.columns.tolist()

# Prepare the DataFrame for plotting
df_scatter = ts_features_df.copy()
df_scatter["Cluster"] = ts_features_df["cluster"]

# Apply log10(x + 1) transformation to the selected variables
for col in vars_scatter:
    df_scatter[col] = np.log10(df_scatter[col] + 1)

sns.pairplot(
    df_scatter[vars_scatter + ["Cluster"]],
    hue="Cluster",
    palette="tab10",
    corner=True,
    diag_kind="hist",
    plot_kws={"alpha": 0.4, "s": 25}
)

plt.suptitle("Scatter Matrix (clustering features, log10(x+1) scale)", y=1.02)
plt.show()

In [ ]:
# Plot cluster size distribution
cluster_counts = ts_features_df['cluster'].value_counts().sort_index()
plt.figure(figsize=(15, 10), dpi=300)
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette='tab10')
plt.xlabel('Cluster', fontsize=14, fontweight='bold')
plt.ylabel('Number of Time Series', fontsize=14, fontweight='bold')
plt.title('Hierarchical Clustering (Ward Linkage) Cluster Size Distribution', fontsize=16, fontweight='bold')
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
for i, count in enumerate(cluster_counts.values):
    plt.text(i, count - count * 0.1, str(count), ha='center', fontsize=10, color='black')
plt.show()

Ward’s method avoided generating single-item clusters. The smallest group, Cluster 6, retained 8 observations, which is more robust than the single observation found in the complete linkage results. While there was still variation in cluster sizes (ranging from n=8 to n=138), a larger proportion of the clusters fell within a ”middle-range” size (between 30 and 60 observations). Because Ward’s method minimises within-cluster variance, it tends to create spherical clusters of equal size. This distribution suggests that Ward linkage offers a more stable segmentation for this specific dataset,avoiding the extreme outliers seen in the complete linkage model.